# Fútbol y apuestas: ¿se le puede ganar a la casa?
## Proyecto final — Gerencia y almacenamiento en Big Data
### Pipeline integral de Big Data con Machine Learning, almacenamiento híbrido y dashboard contenerizado

**Autor:** Juan Felipe Vanegas Silva · Fundación Universitaria Los Libertadores
**Video de sustentación:** *PEGAR AQUÍ EL ENLACE NO LISTADO*

**Dataset:** 238.858 partidos de 38 ligas (2000–2026) con resultados, estadísticas y cuotas de casas de apuestas
(**Football-Data.co.uk**), 273.972 ratings Elo (**ClubElo**) y 656.301 valoraciones de jugadores, partidos y clubes de
**Transfermarkt**, más **1.894.350 minutos de jugadores** (Transfermarkt) y **50.099 partidos de selecciones** (martj42/international_results) para medir fatiga y parones: **3.253.548 registros crudos** de cuatro fuentes públicas.

**Metodología:** CRISP-DM de punta a punta (incluye Modeling y Evaluation).

| # | Sección | Fase CRISP-DM |
|---|---|---|
| 1 | Business Understanding y dataset | Business Understanding |
| 2 | Decisión de arquitectura: pandas o Spark | — |
| 3 | Exploración inicial y calidad de datos | Data Understanding |
| 4 | ETL: limpieza, normalización, enriquecimiento, agregación | Data Preparation |
| 5 | Análisis exploratorio, visualizaciones y OLAP | Data Understanding |
| 6 | Machine Learning | Modeling |
| 7 | Evaluación e interpretación | Evaluation |
| 8 | Insights en MongoDB | Deployment |
| 9 | Reporte PDF | Deployment |
| 10 | Conclusiones | — |

> Este notebook usa las mismas funciones de `src/` que ejecuta el servicio `procesamiento` de Docker, así no hay dos
> versiones del pipeline. Se ejecuta completo dentro del contenedor (ver README, sección *Ejecutar el notebook*).

In [ ]:
import json, os, sys, time, platform
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

import config as C
import descargar_datos, etl, modelo, graficas, mongo_insights, reporte_pdf

# graficas.py fija el backend "Agg" (y el contenedor define MPLBACKEND=Agg) para generar PNG sin pantalla;
# en el notebook se vuelve al backend inline para que las figuras queden embebidas en las salidas.
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
graficas.estilo()
print("Raíz del proyecto:", RAIZ)
print("Python", platform.python_version(), "| pandas", pd.__version__, "| MongoDB en", f"{C.MONGO_HOST}:{C.MONGO_PORT}")

---
## 1. Business Understanding

### 1.1 Problema de negocio
Las casas de apuestas publican cuotas que, en el fondo, son **probabilidades** de cada resultado con un margen para
ellas. Millones de personas apuestan creyendo que "saben más que la casa". La pregunta de negocio es:

> **¿Se puede predecir el resultado de un partido de fútbol con datos públicos tan bien como las casas de apuestas, y
> alcanza eso para obtener un beneficio apostando?**

Responderla sirve a tres públicos: al **apostador** (¿vale la pena?), a **clubes y analistas** (¿qué variables
explican los resultados?) y a un **equipo de datos** (¿cómo se construye un producto de pronósticos reproducible?).

### 1.2 Preguntas específicas
1. **¿Sigue existiendo la ventaja de jugar en casa?** ¿Cómo cambió en 26 años y qué pasó con los estadios vacíos (COVID-19)?
2. **¿Qué tan eficientes son las casas de apuestas?** ¿Están bien calibradas sus probabilidades y cuánto margen cobran?
3. **¿Qué variables explican el resultado?** Elo, forma reciente, valor de la plantilla, descanso, liga…
4. **¿Pesa el dinero?** ¿Gana más quien tiene la plantilla más cara (Transfermarkt)?
5. **¿Puede un modelo igualar a la casa** y obtener ROI positivo con una estrategia de apuestas de valor?
6. **¿Pesan la fatiga y los parones de selecciones?** Si un equipo vuelve de una fecha FIFA con muchos titulares que viajaron con su selección (p. ej. sudamericanos del Real Madrid), ¿rinde peor de lo que espera la casa?

### 1.3 Tarea de Machine Learning y criterio de éxito
- **Tarea:** clasificación multiclase del resultado (**Local / Empate / Visitante**) con información disponible
  **antes** del partido.
- **Líneas base:** (a) *siempre gana el local* (la clase mayoritaria) y (b) *gana el favorito de la casa*.
- **Criterios de éxito definidos antes de modelar:**
  1. Superar la línea base ingenua en **al menos 10 % de F1 macro**.
  2. Lograr un **log-loss igual o menor que el de las probabilidades de la casa** (el mercado es el rival real).
  3. *(Deseable)* ROI positivo en una simulación de apuestas sobre temporadas no vistas.

### 1.4 Descripción del dataset y evidencia de la descarga
Los archivos grandes **no** van en la entrega: `src/descargar_datos.py` los deja en `data/raw/` y registra URL, fecha,
filas, columnas y peso (en disco y en memoria) en `data/raw/registro_descarga.json`.

| Fuente | Qué aporta | Por qué |
|---|---|---|
| Football-Data.co.uk (vía repositorio `xgabora/Club-Football-Match-Data`) | Resultados, estadísticas y cuotas de Bet365 y de ~17 casas | Fuente de referencia del análisis deportivo de apuestas |
| ClubElo | Ratings Elo quincenales de clubes | Medida histórica de fuerza |
| Transfermarkt (`dcaribou/transfermarkt-datasets`) | Partidos con id de club y valor de mercado de jugadores | Dinero/calidad individual; obliga a homologar nombres entre fuentes |
| Transfermarkt: `appearances` y `players` | Minutos de cada jugador en cada partido de todas las competiciones; nacionalidad e internacionalidades | Fatiga: descanso real (con Champions y copa) y carga de minutos |
| martj42/international_results | Todos los partidos de selecciones (eliminatorias, Nations League, amistosos) hasta nov-2026 | Parones: quién jugó con su selección y si cruzó de continente |

> El sitio oficial football-data.co.uk cerraba la conexión desde el equipo de trabajo (ver README, *Problemas de
> infraestructura*), por eso se usa el espejo público en GitHub que reúne los mismos CSV de Football-Data.co.uk.

In [ ]:
registro = descargar_datos.main(forzar=False)
tabla_fuentes = pd.DataFrame(registro).T[["fuente", "url", "fecha_descarga", "filas", "columnas", "peso_disco_mb", "peso_memoria_mb"]]
display(tabla_fuentes)
print(f"TOTAL: {tabla_fuentes['filas'].sum():,} filas crudas | {tabla_fuentes['peso_disco_mb'].sum():.1f} MB en disco | "
      f"{tabla_fuentes['peso_memoria_mb'].sum():.1f} MB en memoria")

---
## 2. Decisión de arquitectura: ¿pandas o Spark?

Se retoman los criterios de la Actividad 4: **volumen frente a la RAM**, **tipo de trabajo** (por fila o por archivo),
**costo de arranque** y **complejidad de despliegue**. Primero, los recursos reales y los tiempos de pandas medidos aquí
mismo (dentro del contenedor):

In [ ]:
def ram_total_gb():
    try:
        import psutil
        return psutil.virtual_memory().total / 1e9
    except ImportError:
        try:
            with open("/proc/meminfo") as f:
                return int(f.readline().split()[1]) / 1e6
        except OSError:
            return float("nan")

limite_contenedor = None
for ruta in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    if os.path.exists(ruta):
        valor = open(ruta).read().strip()
        limite_contenedor = None if valor == "max" or int(valor) > 1e15 else int(valor) / 1e9

t0 = time.perf_counter(); partidos_crudos = pd.read_csv(C.DATA_RAW / "Matches.csv", low_memory=False); t_csv = time.perf_counter() - t0
t0 = time.perf_counter(); valoraciones = pd.read_csv(C.DATA_RAW / "player_valuations.csv.gz"); t_val = time.perf_counter() - t0
mem_crudo = partidos_crudos.memory_usage(deep=True).sum() / 1e6
recursos = pd.Series({
    "CPU lógicos": os.cpu_count(), "RAM del equipo (GB)": round(ram_total_gb(), 1),
    "Límite de memoria del contenedor (GB)": limite_contenedor or "sin límite",
    "Matches.csv: filas": f"{len(partidos_crudos):,}", "Matches.csv: lectura (s)": round(t_csv, 2),
    "Matches.csv: memoria (MB)": round(mem_crudo, 1),
    "player_valuations: filas": f"{len(valoraciones):,}", "player_valuations: lectura (s)": round(t_val, 2),
})
display(recursos.to_frame("valor"))

**Benchmark pandas vs. Spark sobre los mismos datos.** Spark **no** está en la imagen de Docker (es la decisión que
se justifica aquí), así que el benchmark completo se ejecuta en el equipo del autor con `python src/benchmark_spark.py`
(Spark 4.2, `local[*]`, `spark.driver.memory=4g`, `spark.sql.shuffle.partitions = 2 × núcleos`) y su resultado queda
en `reports/benchmark_pandas_spark.json`:

In [ ]:
ruta_bench = C.REPORTS / "benchmark_pandas_spark.json"
if ruta_bench.exists():
    bench = json.loads(ruta_bench.read_text(encoding="utf-8"))
    print("Máquina:", bench["maquina"], "| fecha:", bench["fecha"])
    filas = []
    for operacion in ["lectura_csv_partidos_s", "agregacion_liga_anio_s", "agregacion_y_join_valoraciones_s"]:
        filas.append({"operación": operacion, "pandas (s)": bench["pandas"].get(operacion),
                      "Spark (s)": bench.get("spark", {}).get(operacion, "—")})
    filas.append({"operación": "arranque de la sesión", "pandas (s)": 0, "Spark (s)": bench.get("spark", {}).get("arranque_sesion_s", "—")})
    display(pd.DataFrame(filas))
    if "error" in bench.get("spark", {}):
        print("Spark no pudo arrancar en esa ejecución:", bench["spark"]["error"][:200])
else:
    print("Aún no se ha ejecutado src/benchmark_spark.py en el equipo del autor; se usan solo las mediciones de pandas.")

**Decisión: pandas.** Con datos propios:

| Criterio | Evidencia | Conclusión |
|---|---|---|
| Volumen vs. RAM | ~3,25 M filas crudas (1,9 M de apariciones de jugadores); el dataset limpio ocupa ~**80 MB** en memoria con tipos optimizados, menos del 1 % de la RAM | Cabe de sobra en un proceso: no es Big Data por volumen |
| Tiempos | Leer el CSV más grande toma ~1–2 s y las agregaciones fracciones de segundo | No hay cuello de botella que paralelizar |
| Costo de arranque | Una SparkSession tarda varios segundos solo en arrancar (Actividad 4) y añade JVM | Para este volumen Spark sería más lento de punta a punta |
| Tipo de trabajo | El cálculo más pesado es el **Elo secuencial** (cada partido depende del anterior) | Es intrínsecamente secuencial: Spark no lo acelera |
| Despliegue | Spark en Docker exige Java (~300 MB más en la imagen) y configuración de memoria | Más complejidad sin beneficio |

**Técnicas de pandas para manejar el volumen:** tipos optimizados (`category`, `Int8`, `float32`: el dataset baja de
~91 MB a ~37 MB tras la normalización), lectura de CSV comprimidos `.gz` sin descomprimir, **Parquet** columnar para
todas las salidas (21 MB frente a 45 MB del CSV, y se lee en ~0,15 s) y lectura selectiva de columnas en el dashboard.
Si el proyecto creciera a datos por evento (millones de filas por temporada, p. ej. tracking), la decisión cambiaría a Spark.

---
## 3. Data Understanding: exploración inicial (antes de limpiar)
### 3.1 Estructura, tipos, nulos y duplicados

In [ ]:
print("Forma:", partidos_crudos.shape)
print("Tipos:", partidos_crudos.dtypes.value_counts().to_dict())
nulos = (partidos_crudos.isna().mean() * 100).round(2).sort_values(ascending=False)
display(nulos[nulos > 0].to_frame("% nulos").T)
print("Filas duplicadas exactas:", partidos_crudos.duplicated().sum())
print("Duplicados por (liga, fecha, local, visitante):", partidos_crudos.duplicated(["Division", "MatchDate", "HomeTeam", "AwayTeam"]).sum())
display(partidos_crudos.head(3))

In [ ]:
display(partidos_crudos.describe().T[["count", "mean", "std", "min", "50%", "max"]])

Lo que salta a la vista en `describe()`: **cuotas de 0** (imposibles: una cuota paga al menos lo apostado), un
**hándicap de −99,9**, un partido con **145 faltas** de un equipo, **9 expulsados** en otro y cuotas máximas de **301**.
Además, el **Elo falta en el 37 %** de los partidos (ligas que ClubElo no cubre) y las estadísticas de juego en ~49 %.

### 3.2 Tabla de problemas de calidad detectados

In [ ]:
diagnostico = etl.diagnostico_calidad(partidos_crudos)
display(diagnostico)
tm_partidos = pd.read_csv(C.DATA_RAW / "games.csv.gz", low_memory=False)
display(etl.diagnostico_transfermarkt(valoraciones, tm_partidos))

In [ ]:
# Evidencia concreta de algunos problemas
print("Partidos sin resultado (suspendidos o no jugados):")
display(partidos_crudos[partidos_crudos["FTResult"].isna()][["Division", "MatchDate", "HomeTeam", "AwayTeam", "FTHome", "FTAway", "OddHome"]])
print("Hándicap imposible y cuotas en cero:")
display(partidos_crudos[(partidos_crudos["HandiSize"].abs() > 10) | (partidos_crudos["OddHome"] < 1.01)]
        [["Division", "MatchDate", "HomeTeam", "AwayTeam", "OddHome", "OddDraw", "OddAway", "HandiSize"]].head(8))

Hallazgo con historia: el partido brasileño sin resultado es **Chapecoense vs. Atlético-MG del 12/11/2016**, que no se
jugó tras la tragedia aérea del Chapecoense, y el griego es el **Panathinaikos vs. Olympiakos de 2019**, suspendido.

**Fuga de información detectada:** las columnas `C_*` del espejo son probabilidades de clúster que su autor calculó con
las **estadísticas del mismo partido** (tiros, tarjetas). Usarlas para predecir sería hacer trampa: se eliminan.

---
## 4. Data Preparation: ETL con las 4 técnicas

`etl.ejecutar_etl()` aplica en orden **limpieza → normalización → enriquecimiento → agregación** y anota cada paso con
filas/valores **antes, después y afectados**. Se corre una vez (≈ 1–2 min) y luego se muestra la evidencia de cada técnica.

In [ ]:
t0 = time.perf_counter()
resultado_etl = etl.ejecutar_etl(verbose=False)
print(f"ETL completo en {time.perf_counter() - t0:.1f} s")
partidos = resultado_etl["partidos"]
largo = resultado_etl["largo"]
pasos = resultado_etl["registro"].tabla()

### 4.1 Limpieza

In [ ]:
display(pasos[pasos["tecnica"] == "Limpieza"])
print("Mismo partido con hándicap −99,9 ANTES y DESPUÉS de limpiar:")
fila = partidos_crudos[partidos_crudos["HandiSize"] < -20].iloc[0]
despues = partidos[(partidos["fecha"] == fila["MatchDate"]) & (partidos["local"] == fila["HomeTeam"].strip())]
display(pd.DataFrame({"antes": fila[["HandiSize", "HandiHome", "HandiAway"]].values,
                      "después": despues[["handicap", "cuota_handicap_local", "cuota_handicap_visitante"]].iloc[0].values},
                     index=["hándicap", "cuota hándicap local", "cuota hándicap visitante"]))

**Decisión de diseño:** cuando el error está en *una* estadística (p. ej. 145 faltas) se anula solo ese valor y se
conserva el partido, porque su resultado y sus cuotas siguen siendo válidos. Solo se eliminan los 4 partidos sin resultado.

### 4.2 Normalización

In [ ]:
display(pasos[pasos["tecnica"] == "Normalización"])
print("Cuotas → probabilidades sin margen (suman 1):")
display(partidos[["local", "visitante", "cuota_local", "cuota_empate", "cuota_visitante", "margen_casa",
                  "prob_casa_local", "prob_casa_empate", "prob_casa_visitante"]].dropna().tail(4))
print("Temporada unificada: ligas europeas (jul–jun) vs. calendario anual:")
display(partidos.groupby("calendario_anual", observed=True)[["liga", "fecha", "temporada"]].last())

**Homologación de nombres entre fuentes (el reto de usar varios datasets).** Football-Data escribe "Man United" y
Transfermarkt "Manchester United Football Club"; en Grecia, Rusia o Turquía los nombres ni se parecen. En lugar de
emparejar por texto, se cruzan los partidos de ambas fuentes por **liga + fecha (±1 día) + marcador** y a cada nombre se le
asigna el `club_id` con el que más coincide. La similitud de texto queda como control: los casos con similitud muy baja
son correctos (cirílico, nombres oficiales en griego, "Bergen" = Mons en neerlandés).

In [ ]:
mapeo = resultado_etl["mapeo"]
print(f"{len(mapeo)} equipos homologados")
display(mapeo.sort_values("similitud_texto").head(10))

### 4.3 Enriquecimiento: variables derivadas que alimentan el modelo

In [ ]:
display(pasos[pasos["tecnica"] == "Enriquecimiento"])
print("Variables derivadas en un partido de ejemplo:")
ejemplo = partidos.dropna(subset=["log_ratio_valor", "puntos_prom5_local"]).iloc[-1]
display(ejemplo[["fecha", "local", "visitante", "elo_propio_local", "elo_propio_visitante", "diferencia_elo",
                 "puntos_prom5_local", "puntos_prom5_visitante", "ppg_temporada_local", "dias_descanso_local",
                 "valor_plantilla_local", "valor_plantilla_visitante", "log_ratio_valor", "favorito_casa", "resultado"]].to_frame("valor"))

| Variable derivada | Cómo se calcula | Por qué aporta | Cómo se evita la fuga |
|---|---|---|---|
| `diferencia_elo` (Elo propio) | Elo partido a partido para las 38 ligas (K=20, ventaja local 65, multiplicador por goles) | Resume todo el historial de fuerza; **imputa el 37 % de Elo faltante** | Se guarda el Elo **previo** al partido |
| `puntos_prom5_*`, `gf/gc_prom5_*` | Media móvil de los últimos 5 partidos | Forma reciente | `shift(1)`: solo partidos anteriores |
| `ppg_temporada_*` | Puntos por partido acumulados en la temporada | Rendimiento de la temporada en curso | Acumulado menos el partido actual |
| `record_condicion_*` | Puntos en los últimos 5 como local (o como visitante) | Fortaleza específica en casa / fuera | `shift(1)` |
| `dias_descanso_*` | Días desde el partido anterior (tope 30) | Fatiga / rotaciones | Usa fechas pasadas |
| `log_ratio_valor` | log(valor plantilla local) − log(visitante), Transfermarkt | Calidad individual y presupuesto | Valor al **inicio** de la temporada (12 meses previos) |
| `prob_casa_*`, `margen_casa` | Cuotas normalizadas | Lo que "sabe" el mercado antes del partido | Cuotas pre-partido |
| `descanso_real_*` | Días desde el último partido del club en **cualquier** competición (Transfermarkt + liga) | La variable de descanso anterior solo veía la liga y no la Champions de mitad de semana | Partido anterior, ≥ 2 días antes |
| `carga_min14_*` | Minutos jugados por los jugadores del club entre 15 y 2 días antes | Cansancio acumulado | Solo apariciones anteriores |
| `post_paron`, `convocados_*`, `viajeros_*` | Habituales del club (≥ 45 min en los 35 días previos) cuya selección jugó en la fecha FIFA; viajeros = selección de otra confederación que la liga | La hipótesis del profesor: los titulares vuelven cansados del parón | Solo partidos de selecciones entre el partido anterior y este |

**Validación del Elo propio frente a ClubElo** (donde ambos existen):

**Fatiga y parones con el ejemplo del Real Madrid.** Cada fila es el primer partido después de una fecha FIFA: cuántos habituales jugaron con su selección (convocados) y cuántos cruzaron de continente (viajeros).

In [ ]:
rm = partidos[((partidos["local"] == "Real Madrid") | (partidos["visitante"] == "Real Madrid")) & (partidos["post_paron"] == 1)]
display(rm[["fecha", "local", "visitante", "goles_local", "goles_visitante", "descanso_real_local", "convocados_local",
            "viajeros_local", "convocados_visitante", "viajeros_visitante"]].tail(6))
cob = partidos["descanso_real_local"].notna()
print(f"Partidos con datos de fatiga: {cob.sum():,} · primeros partidos tras un parón: {int((partidos['post_paron'] == 1).sum()):,}")
print(f"Partidos donde el descanso real es menor que el de liga (hubo Champions/copa en medio): "
      f"{(partidos.loc[cob, 'descanso_real_local'] < partidos.loc[cob, 'dias_descanso_local']).mean():.1%}")

In [ ]:
v = partidos[["elo_clubelo_local", "elo_clubelo_visitante", "diferencia_elo"]].dropna()
dif_clubelo = v["elo_clubelo_local"] - v["elo_clubelo_visitante"]
print(f"Correlación de la diferencia de Elo propio vs. ClubElo: r = {dif_clubelo.corr(v['diferencia_elo']):.3f} en {len(v):,} partidos")
fig, ax = plt.subplots(figsize=(6, 5))
muestra = v.sample(6000, random_state=42)
ax.scatter(muestra["elo_clubelo_local"] - muestra["elo_clubelo_visitante"], muestra["diferencia_elo"], s=6, alpha=0.25, color=graficas.ACENTO)
ax.set_xlabel("Diferencia de Elo según ClubElo"); ax.set_ylabel("Diferencia de Elo propio")
ax.set_title("El Elo propio replica a ClubElo (r ≈ 0,96)\ny permite imputar las ligas que ClubElo no cubre")
plt.show()

### 4.4 Agregación (cubos OLAP)

In [ ]:
display(pasos[pasos["tecnica"] == "Agregación"])
cubo_lt, cubo_pd, cubo_eq = resultado_etl["cubos"]
display(cubo_lt.tail(5))

### 4.5 Resultado del ETL y guardado en Parquet

In [ ]:
resumen = resultado_etl["resumen"]
display(pd.Series({k: resumen[k] for k in ["filas_crudas_total", "partidos_limpios", "columnas_limpias", "memoria_limpia_mb",
                                            "peso_parquet_mb", "filas_formato_largo", "equipos", "ligas", "fecha_min", "fecha_max",
                                            "equipos_mapeados_transfermarkt", "partidos_con_valor_mercado"]}).to_frame("valor"))
print("Archivos en data/processed:")
for archivo in sorted(C.DATA_PROCESSED.glob("*")):
    print(f"  {archivo.name:42s} {archivo.stat().st_size / 1e6:7.2f} MB")

---
## 5. Análisis exploratorio y visualizaciones

Principios de la Actividad 7 aplicados en todas las gráficas:

| Principio | Cómo se aplica |
|---|---|
| Tipo de gráfico | Líneas para tiempo, barras ordenadas para categorías, histograma para distribución, calibración para relación, mapa de calor para dos dimensiones |
| Paleta | Local, Empate y Visitante tienen **siempre** el mismo color (verde aguamarina, naranja, azul); validada para daltonismo (ΔE CVD ≥ 9,2 en todos los pares). El verde tiene contraste < 3:1 sobre blanco, por eso hay **etiquetas directas** |
| Data-ink ratio | Sin marco superior/derecho, rejilla tenue solo en un eje, sin 3D, sin sombras, sin leyenda redundante |
| Títulos con insight | El título dice el hallazgo con su cifra; los ejes van en español con unidades |

### 5.1 Pregunta 1: ¿sigue existiendo la ventaja local?

In [ ]:
fig, _ = graficas.ventaja_local(partidos); plt.show()

La ventaja local cae de forma sostenida y el **valle de 2020/21** (estadios vacíos) es la mejor evidencia de que buena
parte de la ventaja es el público. Tras el regreso de la afición se recupera, pero no a los niveles de los 2000.

### 5.2 Distribución de goles

In [ ]:
fig, _ = graficas.distribucion_goles(partidos); plt.show()

Los goles siguen casi una Poisson: hay un poco más de 0–0 y de partidos con muchos goles de lo esperado (sobredispersión leve), algo que un modelo de goles debería tener en cuenta.

In [ ]:
fig, _ = graficas.goles_por_liga(partidos); plt.show()

### 5.3 Pregunta 2: ¿qué tan eficientes son las casas?

In [ ]:
fig, _ = graficas.calibracion_casa(partidos); plt.show()
fig, _ = graficas.margen_casa(partidos); plt.show()

Las probabilidades implícitas de Bet365 caen sobre la diagonal: el mercado está **calibrado**. Y el margen bajó a la
mitad en 25 años; hoy la casa cobra menos, pero sigue cobrando, lo que obliga a cualquier apostador a acertar más que la casa
solo para empatar.

### 5.4 Pregunta 4: ¿pesa el dinero?

In [ ]:
fig, _ = graficas.valor_mercado(partidos); plt.show()

### 5.5 Pregunta 6: ¿pesan la fatiga y los parones?
Se mide **contra lo que esperaba la casa**: victorias locales reales menos la probabilidad implícita. Si el mercado ya descuenta la fatiga, los puntos quedan cerca de 0.

In [ ]:
fig, _ = graficas.fatiga_paron(partidos); plt.show()

Los puntos se mueven en la dirección esperada (el equipo con más viajeros rinde algo peor), pero los intervalos de confianza cruzan el cero: la fatiga de los viajes **existe, pero el mercado ya la tiene en el precio**.

### 5.6 Agregaciones multidimensionales (OLAP)
Los mismos cubos alimentan los filtros del dashboard (panel flotante de *Opciones* y explorador OLAP de la pestaña *Datos*).

**Cubo 1: país × década × nivel × resultado** (mapa de calor):

In [ ]:
fig, _ = graficas.mapa_pais_decada(partidos); plt.show()

In [ ]:
# SLICE: un solo valor de una dimensión (temporada 2025/26) sobre el cubo liga × temporada
slice_2025 = cubo_lt[(cubo_lt["temporada_inicio"] == 2025) & (cubo_lt["nivel"] == 1)]
display(slice_2025[["pais", "liga", "partidos", "pct_local", "pct_empate", "pct_visitante", "goles_prom", "margen_casa_prom", "acierto_favorito"]]
        .sort_values("goles_prom", ascending=False).head(10))

# DICE: varias dimensiones a la vez (España e Inglaterra × niveles 1 y 2 × década 2020s)
dice = cubo_pd[cubo_pd["pais"].isin(["España", "Inglaterra"]) & cubo_pd["nivel"].isin([1, 2]) & (cubo_pd["decada"] == "2020s")]
display(dice.pivot_table(index=["pais", "nivel"], columns="resultado", values="pct_dentro_grupo", observed=True).round(1))

In [ ]:
# DRILL-DOWN: continente → país → liga (y ROLL-UP en sentido contrario) sobre los partidos limpios
for nivel_jerarquia in (["continente"], ["continente", "pais"], ["continente", "pais", "liga"]):
    tabla = (partidos.groupby(nivel_jerarquia, observed=True)
             .agg(partidos=("resultado", "size"), goles=("total_goles", "mean"),
                  pct_local=("resultado", lambda s: (s == "Local").mean() * 100)).round(2))
    print("Nivel:", " → ".join(nivel_jerarquia), f"({len(tabla)} grupos)")
    display(tabla.sort_values("partidos", ascending=False).head(6))

In [ ]:
# Cubo 3: equipo × temporada (tabla de posiciones) — líderes de las 5 grandes ligas 2025/26
lideres = cubo_eq[(cubo_eq["temporada_inicio"] == 2025) & (cubo_eq["posicion"] == 1) & cubo_eq["liga_codigo"].isin(["E0", "SP1", "I1", "D1", "F1"])]
display(lideres[["liga", "equipo", "partidos", "victorias", "empates", "derrotas", "gf", "gc", "puntos", "valor_plantilla_meur"]])

---
## 6. Modeling

### 6.1 Preparación de variables y cómo se evitó la fuga de información
- **Solo información previa al pitazo:** se excluyen goles, tiros, tarjetas, córners, resultado al descanso y las columnas
  `C_*`. Las variables de forma usan `shift(1)`, el Elo es el previo al partido y el valor de plantilla es el del inicio
  de la temporada.
- **Partición temporal, nunca aleatoria:** entrenamiento hasta 30/06/2022, validación 2022/23 y prueba desde el
  01/07/2023. Barajar mezclaría partidos del futuro en el entrenamiento.
- **Preprocesamiento dentro de un `Pipeline`:** imputación (mediana + indicador de faltante), escalado y codificación
  (one-hot u ordinal de la liga) se ajustan **solo** con entrenamiento.
- **Selección sin mirar prueba:** el mejor algoritmo se elige por log-loss en validación y el umbral de empates también
  se elige en validación, con el modelo que no vio validación. Después se re-entrena con entrenamiento + validación.

### 6.2 Modelos
Dos líneas base y tres algoritmos, cada uno con dos conjuntos de variables:
**sin cuotas** (fútbol puro: Elo, forma, valor de plantilla, descanso, liga) y **con cuotas** (lo anterior más las
probabilidades del mercado). Comparar ambos mide cuánto sabe el mercado más allá de los datos públicos.

| Modelo | Por qué |
|---|---|
| Regresión logística multinomial | Lineal, interpretable y bien calibrada |
| Random Forest | No lineal y robusto, poco ajuste de hiperparámetros |
| LightGBM | Boosting de gradiente con parada temprana en validación; maneja faltantes de forma nativa |

El entrenamiento completo tarda ≈ 4–5 minutos (Random Forest es el más lento).

In [ ]:
t0 = time.perf_counter()
metricas = modelo.ejecutar_modelado(partidos)
print(f"Modelado en {time.perf_counter() - t0:.1f} s")

---
## 7. Evaluation
### 7.1 Métricas en prueba frente a las líneas base

In [ ]:
filas = []
for nombre, r in metricas["lineas_base"].items():
    filas.append({"modelo": f"LÍNEA BASE · {nombre}", **{k: r["prueba"][k] for k in ["accuracy", "balanced_accuracy", "f1_macro", "log_loss", "brier", "roc_auc_ovr"]}})
for conjunto, res in metricas["resultados"].items():
    for nombre, r in res.items():
        if not nombre.startswith("_"):
            filas.append({"modelo": f"{nombre} · {conjunto}", **{k: r["prueba"][k] for k in ["accuracy", "balanced_accuracy", "f1_macro", "log_loss", "brier", "roc_auc_ovr"]}})
tabla_metricas = pd.DataFrame(filas).set_index("modelo")
display(tabla_metricas.style.format("{:.4f}").highlight_min(subset=["log_loss", "brier"], color="#cdeee0").highlight_max(subset=["accuracy", "f1_macro"], color="#cdeee0"))
for conjunto in ["con_cuotas", "sin_cuotas"]:
    m = metricas["resultados"][conjunto]["_mejor"]
    print(f"Mejor {conjunto}: {m['modelo']} | umbral de empate {m['umbral_empate']} | F1 macro con umbral {m['prueba_con_umbral']['f1_macro']:.4f}")
display(pd.Series(metricas["criterios_exito"]).to_frame("criterios de éxito"))

In [ ]:
fig, _ = graficas.comparacion_modelos(metricas); plt.show()
fig, _ = graficas.matriz_confusion(metricas); plt.show()
mejor = metricas["resultados"]["con_cuotas"]["_mejor"]
display(pd.DataFrame(mejor["prueba_con_umbral"]["por_clase"]).T)

**Lectura de las métricas.**
- **Criterio 1 (F1 macro ≥ +10 % sobre la ingenua): cumplido de sobra.** "Siempre local" tiene F1 macro ≈ 0,20 porque
  nunca acierta empates ni visitantes.
- **Criterio 2 (log-loss ≤ casa): empate técnico.** Con cuotas, los tres algoritmos quedan a menos de 0,001 del mercado
  (diferencias en la cuarta cifra decimal; la tabla de criterios dice si en esta corrida queda justo por encima o por debajo).
  El mercado ya incorpora casi toda la información pública.
- **Sin cuotas**, el mejor modelo queda ~1 punto de accuracy por debajo de la casa usando solo datos públicos y gratuitos.
- La **accuracy ronda el 50 %**: el fútbol es un deporte de pocos goles y mucha varianza, y ningún modelo (ni la casa)
  supera ~52 % en un dataset tan amplio.
- El **empate** es la clase difícil: casi nunca es el resultado más probable. El umbral ajustado en validación sube su
  recall de ~0 % a ~25 % y mejora el F1 macro, a cambio de un poco de accuracy.

### 7.2 Interpretación en lenguaje de negocio

In [ ]:
fig, _ = graficas.importancia(metricas, "sin_cuotas"); plt.show()
display(pd.DataFrame(metricas["importancias"]["con_cuotas"]).head(8)[["etiqueta", "importancia", "desviacion"]])

**Ablación: ¿aportan las 12 variables de fatiga y parón?** Mismo LightGBM, entrenado con y sin ellas. Mejora = log-loss sin fatiga − con fatiga (positivo = la fatiga ayuda), en toda la prueba y solo en los partidos tras un parón.

Lectura: en el total de la prueba la mejora es casi nula, porque la mayoría de partidos no vienen de un parón. En los partidos justo después de una fecha FIFA la mejora es muchas veces mayor, incluso en el modelo con cuotas: la fatiga de los convocados aporta información que el mercado no tiene del todo descontada.

In [ ]:
abl = metricas["ablacion_fatiga"]
display(pd.DataFrame({c: {"log-loss sin fatiga": abl[c]["sin_fatiga"]["log_loss"], "log-loss con fatiga": abl[c]["con_fatiga"]["log_loss"],
                          "mejora (todos)": abl[c]["mejora_log_loss"], "mejora (tras parón)": abl[c]["mejora_log_loss_post_paron"]}
                      for c in ["sin_cuotas", "con_cuotas"]}).T.style.format("{:+.5f}"))
print("Partidos de prueba tras un parón:", abl["partidos_post_paron_prueba"])

- **En fútbol puro, la diferencia de Elo es la variable reina**: saber cuánto mejor es un equipo, resumiendo todo su
  historial, explica más que cualquier racha. Le siguen la liga (cada una tiene distinta tasa de empates y de localía) y el
  valor de plantilla (fichajes que el Elo aún no refleja).
- **La forma de los últimos 5 partidos casi no aporta** una vez se conoce el Elo: las rachas se sobrevaloran.
- **Con cuotas, el log-odds de la casa concentra casi todo**: el mercado ya resume Elo, forma, lesiones y noticias. Lo
  que el modelo añade encima es pequeño, pero existe (ver la simulación).

### 7.3 Pregunta 5: ¿alcanza para ganar dinero?
Estrategia de **apuestas de valor** sobre las temporadas de prueba: apostar 1 unidad cuando `p_modelo × cuota − 1 > umbral`,
con la mejor cuota del mercado. La **estrategia de control** usa las probabilidades de la propia casa: si también ganara,
la ventaja vendría solo de buscar la mejor cuota y no del modelo.

In [ ]:
display(pd.DataFrame({k: v["umbral_5pct"] for k, v in metricas["simulacion"].items()}).T
        [["apuestas", "aciertos", "tasa_acierto", "cuota_media", "beneficio_unidades", "roi", "max_caida"]])
fig, _ = graficas.curva_roi(metricas); plt.show()

El modelo **con cuotas** obtiene un ROI levemente positivo con umbrales bajos, mientras el control (probabilidades de la
casa) y el modelo sin cuotas pierden. Es decir, el modelo sí añade algo de información útil al mercado, **pero el margen es
pequeño y volátil**: con umbrales altos se hacen pocas apuestas y el ROI salta por pura varianza. En la práctica, además, las
casas limitan las cuentas ganadoras y las cuotas máximas no siempre están disponibles: es una prueba de concepto, no un
sistema para apostar.

### 7.4 Respuestas a las preguntas de negocio

In [ ]:
grandes = partidos[partidos["liga_codigo"].isin(["E0", "SP1", "I1", "D1", "F1"]) & (partidos["temporada_inicio"] <= 2025)]
local_por_temporada = grandes.groupby("temporada_inicio")["resultado"].apply(lambda s: (s == "Local").mean() * 100)
casa = metricas["lineas_base"]["Favorito de la casa"]["prueba"]
mejor_con = metricas["resultados"]["con_cuotas"]["_mejor"]
mejor_sin = metricas["resultados"]["sin_cuotas"]["_mejor"]
r_con = metricas["resultados"]["con_cuotas"][mejor_con["modelo"]]["prueba"]
r_sin = metricas["resultados"]["sin_cuotas"][mejor_sin["modelo"]]["prueba"]
europa = partidos[(partidos["nivel"] == 1) & (partidos["continente"] == "Europa")].groupby("temporada_inicio")["margen_casa"].mean() * 100
respuestas = [
    ("1. ¿Sigue la ventaja local?", f"Sí, pero menor: {local_por_temporada.loc[2000:2002].mean():.1f} % → {local_por_temporada.loc[2023:2025].mean():.1f} % de victorias locales; {local_por_temporada.loc[2020]:.1f} % con estadios vacíos."),
    ("2. ¿Son eficientes las casas?", f"Muy: calibradas casi sobre la diagonal y con un margen que bajó de {europa.iloc[0]:.1f} % a {europa.loc[:2025].iloc[-1]:.1f} %."),
    ("3. ¿Qué explica el resultado?", f"Ante todo la diferencia de Elo; luego la liga y el valor de plantilla. La forma reciente aporta poco."),
    ("4. ¿Pesa el dinero?", "Sí: el decil con la plantilla relativa más cara gana ~66 % como local vs. ~24 % del decil más barato."),
    ("6. ¿Pesan la fatiga y los parones?", f"En promedio casi nada (mejora de log-loss {metricas['ablacion_fatiga']['sin_cuotas']['mejora_log_loss']:+.5f}), pero justo tras un parón la mejora es {metricas['ablacion_fatiga']['sin_cuotas']['mejora_log_loss_post_paron']:+.5f} sin cuotas y {metricas['ablacion_fatiga']['con_cuotas']['mejora_log_loss_post_paron']:+.5f} con cuotas: informa donde la hipótesis aplica."),
    ("5. ¿Se iguala a la casa? ¿Hay ROI?", f"Con cuotas, log-loss {r_con['log_loss']:.4f} vs. {casa['log_loss']:.4f} (igual). Sin cuotas, {r_sin['accuracy']*100:.1f} % vs. {casa['accuracy']*100:.1f} % de accuracy. ROI con valor ≥ 5 %: {metricas['simulacion']['con_cuotas']['umbral_5pct']['roi']*100:+.2f} %, pequeño y volátil."),
]
display(pd.DataFrame(respuestas, columns=["pregunta", "respuesta"]).style.hide(axis="index"))

---
## 8. Insights en MongoDB
Se construye el documento de análisis con la estructura de la Actividad 1, ampliada con el modelo (`metadata`,
`calidad_datos`, `estadisticas_descriptivas`, `agregaciones`, `modelo`, `insights`, `visualizaciones_generadas`,
`recomendaciones`). Todas las cifras de los insights se calculan a partir de los datos; ninguna está escrita a mano.

In [ ]:
visualizaciones = graficas.generar_todas(partidos, metricas)
documento = mongo_insights.construir_documento(partidos, metricas, resultado_etl["resumen"], visualizaciones)
print("Secciones:", list(documento.keys()))
print(f"{len(documento['insights'])} insights | {len(documento['recomendaciones'])} recomendaciones | {len(visualizaciones)} visualizaciones")
mongo_insights.guardar(documento)
cliente, coleccion = mongo_insights.conectar()
print("Documentos en la colección:", coleccion.count_documents({}))

### 8.1 Consultas a MongoDB

In [ ]:
# Consulta 1: recuperar las métricas del modelo y de las líneas base (proyección)
q1 = coleccion.find_one({"_id": mongo_insights.DOC_ID}, {"_id": 0, "modelo.algoritmo": 1, "modelo.metricas": 1, "modelo.linea_base": 1})
print("Algoritmo:", q1["modelo"]["algoritmo"])
display(pd.DataFrame({"modelo": q1["modelo"]["metricas"], **q1["modelo"]["linea_base"]}).T[["accuracy", "f1_macro", "log_loss"]])

In [ ]:
# Consulta 2: filtrar insights por tema ($unwind + $match + $project)
q2 = list(coleccion.aggregate([
    {"$match": {"_id": mongo_insights.DOC_ID}}, {"$unwind": "$insights"},
    {"$match": {"insights.tema": {"$in": ["Modelo", "Apuestas"]}}},
    {"$project": {"_id": 0, "tema": "$insights.tema", "titulo": "$insights.titulo", "valor": "$insights.valor"}}]))
display(pd.DataFrame(q2))

In [ ]:
# Consulta 3: las 5 variables más importantes ordenadas ($unwind + $sort + $limit)
q3 = list(coleccion.aggregate([
    {"$match": {"_id": mongo_insights.DOC_ID}}, {"$unwind": "$modelo.variables_importantes_sin_cuotas"},
    {"$sort": {"modelo.variables_importantes_sin_cuotas.importancia": -1}}, {"$limit": 5},
    {"$project": {"_id": 0, "variable": "$modelo.variables_importantes_sin_cuotas.etiqueta",
                  "importancia": "$modelo.variables_importantes_sin_cuotas.importancia"}}]))
display(pd.DataFrame(q3))

In [ ]:
# Consulta 4: conteo de insights por tema ($group) y problemas de calidad más frecuentes
display(pd.DataFrame(list(coleccion.aggregate([
    {"$match": {"_id": mongo_insights.DOC_ID}}, {"$unwind": "$insights"},
    {"$group": {"_id": "$insights.tema", "cantidad": {"$sum": 1}}}, {"$sort": {"cantidad": -1}}]))))
display(pd.DataFrame(mongo_insights.consultas_ejemplo(coleccion)["4_problemas_calidad_mas_frecuentes"]))

**Insights y recomendaciones guardados** (leídos de vuelta desde MongoDB):

In [ ]:
guardado = coleccion.find_one({"_id": mongo_insights.DOC_ID}, {"insights": 1, "recomendaciones": 1})
for ins in guardado["insights"]:
    display(Markdown(f"**{ins['id']}. {ins['titulo']}** *({ins['tema']})* — {ins['texto']}"))
for r in guardado["recomendaciones"]:
    display(Markdown(f"**R{r['id']} · {r['dirigida_a']}:** {r['texto']}"))

---
## 9. Reporte PDF generado automáticamente
`reporte_pdf.generar()` **lee el documento desde MongoDB** y arma con ReportLab: portada, resumen del dataset, problemas de
calidad y pasos ETL, tablas de agregación, gráficas de EDA, resultados del modelo, simulación, insights y recomendaciones.

In [ ]:
documento_mongo = mongo_insights.leer()
ruta_pdf = reporte_pdf.generar(documento_mongo)
print(f"PDF: {ruta_pdf.relative_to(RAIZ)} ({ruta_pdf.stat().st_size / 1e6:.2f} MB)")
import pypdfium2 as pdfium
pdf = pdfium.PdfDocument(str(ruta_pdf))
print("Páginas:", len(pdf))
fig, ejes = plt.subplots(1, 3, figsize=(13, 6))
for eje, pagina in zip(ejes, [0, 2, 9 if len(pdf) > 9 else len(pdf) - 1]):
    eje.imshow(pdf[pagina].render(scale=0.9).to_pil()); eje.axis("off"); eje.set_title(f"Página {pagina + 1}", loc="center")
plt.show()

---
## 10. Conclusiones y aprendizajes

1. **El mercado de apuestas es muy eficiente.** Sus probabilidades están calibradas y un modelo con datos públicos solo
   logra igualarlo cuando *usa* sus cuotas. Sin ellas, se queda a ~1 punto de accuracy, lo cual es meritorio para datos gratuitos.
2. **La ventaja local existe pero se encoge**, y los estadios vacíos de 2020/21 demostraron que la afición es buena parte de ella.
3. **Elo y dinero explican el fútbol mejor que las rachas.** La forma reciente casi no aporta encima del Elo.
4. **La fatiga de los parones pesa justo después de la fecha FIFA.** Integrar 1,9 M de apariciones de jugadores y 50 mil partidos de selecciones permitió medir convocados y viajes intercontinentales; en promedio el modelo apenas mejora, pero en los partidos tras un parón la mejora es muchas veces mayor.
5. **Ganarle a la casa es posible solo en el margen**, con la mejor cuota disponible y aceptando mucha varianza.
6. **Integrar varias fuentes fue el mayor reto de ETL:** homologar nombres de equipos sin una llave común se resolvió con
   evidencia de partidos (liga + fecha + marcador) en lugar de comparar textos.

**Qué funcionó:** la partición temporal y la disciplina anti-fuga (los resultados de prueba son creíbles), el Elo propio
para imputar ligas sin ClubElo y la arquitectura de tres servicios que separa procesamiento, base de datos y app.

**Qué no funcionó o haría distinto:** predecir empates sigue siendo débil (probaría un modelo de goles tipo Dixon-Coles);
las estadísticas de juego (tiros, xG) faltan en la mitad de las ligas y no se pudieron usar como forma reciente; y con más
tiempo agregaría cuotas de cierre y apertura para medir el movimiento del mercado.

Los problemas de infraestructura encontrados (CDN que bloquea `Python-urllib`, servidor que corta la conexión, sockets
huérfanos de Docker Desktop, Spark sin sockets AF_UNIX, `log_loss` con columnas desordenadas…) están diagnosticados paso a paso en el **README**.

In [ ]:
# Cierre de conexiones a la base de datos
cliente.close()
print("Conexión a MongoDB cerrada. Notebook completo.")